# Auracare PCOS quiz training
Upload the supplied ZIP. It includes trained artifacts and source snapshots. Use only your trusted package.

Experimental prior-diagnosis labels; not a diagnosis. Read WALKTHROUGH.md.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
destination = Path('/content/auracare_experiment')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (destination / info.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
os.chdir(destination / 'auracare_pcos')
print(Path.cwd())


Install the exact training dependencies. A Colab runtime restart may be required if other packages were imported first.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


Inspect the audit and prepared data. Rerun training to reproduce preprocessing, development selection, calibration and the untouched holdout evaluation.

In [ ]:
subprocess.run([sys.executable, 'train.py'], check=True)


In [ ]:
import json
from IPython.display import display, Markdown, Image
metrics = json.loads(Path('outputs/metrics.json').read_text())
print(json.dumps({'selected_model':metrics['selected_model'],'holdout':metrics['holdout']}, indent=2))
display(Image('outputs/evaluation.png'))


Run software/data integrity tests. These do not constitute clinical validation.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', '-v'], check=True)


Test a synthetic questionnaire response. Edit the example JSON to try other valid inputs. Missing core symptom answers abstain.

In [ ]:
subprocess.run([sys.executable, 'predict.py', '--input', 'examples/quiz_answers.json'], check=True)


In [ ]:
display(Markdown(Path('WALKTHROUGH.md').read_text()))


Optional: download the regenerated outputs and fitted model files.

In [ ]:
import shutil
archive = shutil.make_archive('/content/auracare_retrained', 'zip', root_dir=Path.cwd().parent, base_dir=Path.cwd().name)
files.download(archive)
